
## Step 0 — Environment

This notebook was created and run locally in VS Code using the course `uv`
Python 3.12 environment. The course folder contains the `.venv` environment,
`pyproject.toml`, and `uv.lock` files, confirming that the required local
environment and package setup are available.

In [1]:
import pandas, statsmodels, yfinance, sys

print(
    sys.version.split()[0],
    pandas.__version__,
    statsmodels.__version__,
    yfinance.__version__
)

3.12.14 2.2.3 0.15.0 1.7.0


## Step 1 — Conventions

Return type: simple monthly returns from adjusted month-end close downloaded
with `yfinance` using `auto_adjust=True`.

Frequency: monthly; annualization will use ×12 for mean returns and alphas,
and ×√12 for volatility.

Risk-free: monthly `RF` from the Ken French Data Library’s Fama/French
3 Factors file.

Window: 2015-07 to 2025-06, 120 months.

Market series: Fama/French `Mkt-RF` for factor regressions and `SPY` as the
investable market benchmark.

Equal-weight: rebalanced monthly across 11 securities, with a weight of 1/11
in each security.

Standard deviation: sample standard deviation (`ddof=1`).

Panel: BAC, C, CFG, IBM, JPM, KEY, PNC, TFC, USB, WFC, and ZION.

## Step 2 — Build the Excess-Return Panel

I retain the bank-focused panel from Lab 2 but exclude Credit Suisse, First
Republic, Signature Bank, and Silicon Valley Bank because they do not provide
complete price histories through June 2025. CMA was replaced with CFG because
yfinance returned zero rows for CMA despite its CRSP coverage in Lab 2.

In [2]:
import pandas as pd
import yfinance as yf
from pathlib import Path

SECURITIES = [
    "BAC", "C", "CFG", "IBM", "JPM", "KEY",
    "PNC", "TFC", "USB", "WFC", "ZION"
]

TICKERS = SECURITIES + ["SPY"]

WINDOW_START = "2015-07"
WINDOW_END = "2025-06"
EXPECTED_MONTHS = 120

# Start one month early to calculate the July 2015 return
PRICE_START = "2015-06-01"
PRICE_END = "2025-07-01"

ASSET_COLUMNS = TICKERS

In [3]:
# This works whether the notebook is running from the main folder or lab3
ff_file = Path("lab3/F-F_Research_Data_Factors.csv")

if not ff_file.exists():
    ff_file = Path("F-F_Research_Data_Factors.csv")

assert ff_file.exists(), "Put F-F_Research_Data_Factors.csv in the lab3 folder."

lines = ff_file.read_text(encoding="latin1").splitlines()

# 1. Parses the monthly block only
header_row = next(
    i for i, line in enumerate(lines)
    if line.strip().startswith(",Mkt-RF")
)

end_row = next(
    i for i in range(header_row + 1, len(lines))
    if not lines[i].strip()
)

ff = pd.read_csv(
    ff_file,
    skiprows=header_row,
    nrows=end_row - header_row - 1
)

ff.columns = ff.columns.str.strip()
ff = ff.rename(columns={ff.columns[0]: "Month"})

# 3. Keys the rows on a monthly period
ff["Month"] = pd.to_datetime(
    ff["Month"].astype(str),
    format="%Y%m"
).dt.to_period("M")

# 2. Divides percentage returns by 100
ff = (
    ff.set_index("Month")[["Mkt-RF", "SMB", "HML", "RF"]]
    .apply(pd.to_numeric, errors="coerce")
    .div(100)
    .loc[WINDOW_START:WINDOW_END]
)

ff.head()

,Mkt-RF,SMB,HML,RF
Month,,,,
2015-07,0.0157,-0.0417,-0.0416,0.0
2015-08,-0.0602,0.0037,0.0275,0.0
2015-09,-0.0306,-0.0263,0.0055,0.0
2015-10,0.0775,-0.0191,-0.0046,0.0
2015-11,0.0057,0.0358,-0.0043,0.0


In [4]:
# Confirm the expected Fama/French columns were parsed
print(ff.columns.tolist())

['Mkt-RF', 'SMB', 'HML', 'RF']


In [5]:
prices = yf.download(
    TICKERS,
    start=PRICE_START,
    end=PRICE_END,
    auto_adjust=True,
    progress=False
)["Close"]

monthly_prices = prices.resample("ME").last()

raw_monthly_returns = monthly_prices.pct_change(fill_method=None)
raw_monthly_returns.index = raw_monthly_returns.index.to_period("M")
raw_monthly_returns = raw_monthly_returns.loc[WINDOW_START:WINDOW_END]

raw_monthly_returns.head()

Ticker,BAC,C,CFG,IBM,JPM,KEY,PNC,SPY,TFC,USB,WFC,ZION
Date,,,,,,,,,,,,
2015-07,0.050529,0.059190,-0.041719,-0.004119,0.017974,-0.011984,0.031916,0.022589,-0.000992,0.041705,0.028983,-0.017328
2015-08,-0.086130,-0.085187,-0.047948,-0.079463,-0.064643,-0.069094,-0.071909,-0.060950,-0.076991,-0.063260,-0.072436,-0.068375
2015-09,-0.043442,-0.072363,-0.038678,-0.019744,-0.048830,-0.053130,-0.021071,-0.025516,-0.035753,-0.025679,-0.037127,-0.050345
2015-10,0.077022,0.072739,0.018441,-0.033731,0.061401,-0.045349,0.017668,0.085060,0.043539,0.028530,0.054333,0.044663
2015-11,0.038736,0.017303,0.100385,0.004624,0.037821,0.061610,0.058166,0.003655,0.046883,0.040541,0.024728,0.043486


In [6]:
# Join raw monthly security returns to the monthly Fama/French factor data
# An inner join keeps only months available in both sources
joined = raw_monthly_returns.join(ff, how="inner")

# Define the 120 monthly periods declared in Step 1
expected_index = pd.period_range(WINDOW_START, WINDOW_END, freq="M")

# Confirm that no months were lost during the join
assert joined.index.equals(expected_index), (
    f"Expected {WINDOW_START} to {WINDOW_END}, but the joined data runs "
    f"{joined.index.min()} to {joined.index.max()} with {len(joined)} months."
)

# Check that each security and SPY has a valid return in every joined month
missing_by_column = joined[ASSET_COLUMNS].isna().sum()

assert (missing_by_column == 0).all(), (
    "Missing monthly returns:\n"
    + missing_by_column[missing_by_column > 0].to_string()
)

# Confirm the observed month count matches the Step 1 convention
assert len(joined) == EXPECTED_MONTHS

print(f"joined: {joined.shape}    {joined.index.min()} -> {joined.index.max()}")

# Preserve the raw joined frame for later wealth and CAGR calculations
# Create a separate frame for excess-return analysis
excess_returns = joined.copy()

# Subtract RF from stocks and SPY only
excess_returns[ASSET_COLUMNS] = joined[ASSET_COLUMNS].sub(
    joined["RF"],
    axis=0
)

# Do not subtract RF from Mkt-RF
show_columns = ["JPM", "SPY", "Mkt-RF", "RF"]


print("First row:")
display(joined.loc[[joined.index[0]], show_columns].round(4))

print("Last row:")
display(joined.loc[[joined.index[-1]], show_columns].round(4))

joined: (120, 16)    2015-07 -> 2025-06
First row:


,JPM,SPY,Mkt-RF,RF
Date,,,,
2015-07,0.018,0.0226,0.0157,0.0


Last row:


,JPM,SPY,Mkt-RF,RF
Date,,,,
2025-06,0.0981,0.0514,0.0486,0.0034


## Open data-quality issue — survivorship and panel coverage

The 120-month return panel is a continuous-coverage, investable subset rather
than the original 15-security Lab 2 bank-stress universe. Credit Suisse, First
Republic, Signature Bank, and Silicon Valley Bank were excluded because they
did not remain available through the June 2025 endpoint. CMA also returned zero
rows from yfinance, so CFG was substituted.

This construction can bias the equal-weight panel’s later alpha upward because
the omitted firms include distressed institutions that exited the sample. The
panel can therefore support analysis of the surviving/continuous-coverage
securities, but not a headline performance or alpha claim about the complete
original Lab 2 universe without a delisting-return or CRSP-based treatment.

## Checkpoint 4 — Stopping point

I completed Steps 0–2: configured the local VS Code environment, stated the
return conventions, downloaded and parsed the monthly Fama/French factors, and
built and validated the 120-month raw and excess-return panel. I stopped after
confirming the joined panel runs from 2015-07 through 2025-06 with no missing
monthly returns.

## AI Disclosure

ChatGPT (OpenAI) was used to help set up VS Code and the local Python
environment, explain the assignment instructions, troubleshoot file-path and
`yfinance` errors, and draft/review Python code for parsing the Fama/French
file, calculating monthly returns, and validating the joined panel. I selected
the bank-focused panel, ran the code locally, reviewed the outputs, and changed
CMA to CFG after `yfinance` returned no CMA data.

Codex CLI was used to confirm that it could access the course folder by listing
its files. It was not used to run the Lab 3 analysis.

In [7]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

MONTHS_PER_YEAR = 12

# All stock columns, excluding the benchmark and Fama-French factor columns
exclude_cols = ["SPY", "Mkt-RF", "SMB", "HML", "RF"]
security_cols = [col for col in joined.columns if col not in exclude_cols]

print("Securities used in the equal-weight panel:")
print(security_cols)
print("Number of securities:", len(security_cols))

asset_returns = joined[security_cols].copy()
asset_excess = excess_returns[security_cols].copy()
mkt_excess = joined["Mkt-RF"].copy()

ew_return = asset_returns.mean(axis=1).rename("EW panel")
ew_excess = asset_excess.mean(axis=1).rename("EW panel")

returns_all = pd.concat([asset_returns, ew_return], axis=1)
excess_all = pd.concat([asset_excess, ew_excess], axis=1)

assert returns_all.isna().sum().sum() == 0
assert excess_all.isna().sum().sum() == 0

print(returns_all.shape, returns_all.index.min(), "to", returns_all.index.max())

Securities used in the equal-weight panel:
['BAC', 'C', 'CFG', 'IBM', 'JPM', 'KEY', 'PNC', 'TFC', 'USB', 'WFC', 'ZION']
Number of securities: 11
(120, 12) 2015-07 to 2025-06


In [8]:
def cagr(monthly_returns):
    n_months = len(monthly_returns)
    return (1 + monthly_returns).prod() ** (MONTHS_PER_YEAR / n_months) - 1

step3 = pd.DataFrame(index=returns_all.columns)

# CAGR uses ordinary total returns
step3["CAGR"] = returns_all.apply(cagr)

# Volatility is based on monthly excess returns, annualized
step3["ann vol"] = excess_all.std(ddof=1) * np.sqrt(MONTHS_PER_YEAR)

# Correct Week 3 Sharpe: annualized arithmetic mean excess return / annualized volatility
step3["Sharpe (excess, arithmetic)"] = (
    excess_all.mean() * MONTHS_PER_YEAR
) / step3["ann vol"]

# Shortcut for comparison only
step3["CAGR/vol, rf=0"] = step3["CAGR"] / step3["ann vol"]

step3_display = step3.copy()
for col in ["CAGR", "ann vol"]:
    step3_display[col] = step3_display[col].map(lambda x: f"{x:.2%}")
for col in ["Sharpe (excess, arithmetic)", "CAGR/vol, rf=0"]:
    step3_display[col] = step3_display[col].map(lambda x: f"{x:.3f}")

step3_display

,CAGR,ann vol,"Sharpe (excess, arithmetic)","CAGR/vol, rf=0"
BAC,13.17%,29.75%,0.502,0.443
C,7.39%,32.59%,0.326,0.227
CFG,8.91%,36.45%,0.372,0.245
IBM,11.29%,23.97%,0.489,0.471
JPM,18.86%,24.50%,0.755,0.770
KEY,5.63%,34.50%,0.281,0.163
PNC,10.33%,27.02%,0.430,0.382
TFC,4.66%,28.38%,0.241,0.164
USB,4.03%,26.56%,0.212,0.152
WFC,6.59%,29.33%,0.302,0.225


### Step 3 Interpretation

Rank changes occur between BAC and IBM; among C, WFC, and ZION; and between KEY and TFC when comparing the two measures. I will use the Sharpe ratio based on arithmetic excess returns for the rest of the course because it is the Week 3 standard and properly accounts for the risk-free rate.

In [9]:
X = sm.add_constant(mkt_excess)

jensen_rows = []
models = {}

for name in excess_all.columns:
    model = sm.OLS(excess_all[name], X).fit()
    models[name] = model
    
    jensen_rows.append({
        "security": name,
        "α/yr": model.params["const"] * MONTHS_PER_YEAR,
        "t(α)": model.tvalues["const"],
        "β": model.params["Mkt-RF"],
        "t(β)": model.tvalues["Mkt-RF"],
        "R²": model.rsquared,
        "resid vol": np.sqrt(model.mse_resid * MONTHS_PER_YEAR)
    })

step4 = pd.DataFrame(jensen_rows).set_index("security")

step4_display = step4.copy()
for col in ["α/yr", "resid vol"]:
    step4_display[col] = step4_display[col].map(lambda x: f"{x:.2%}")
for col in ["t(α)", "β", "t(β)", "R²"]:
    step4_display[col] = step4_display[col].map(lambda x: f"{x:.3f}")

step4_display

,α/yr,t(α),β,t(β),R²,resid vol
security,,,,,,
BAC,-1.97%,-0.317,1.415,12.920,0.586,19.22%
C,-7.70%,-1.112,1.532,12.571,0.573,21.40%
CFG,-3.91%,-0.432,1.461,9.169,0.416,27.97%
IBM,1.10%,0.177,0.889,8.080,0.356,19.31%
JPM,5.04%,0.942,1.126,11.936,0.547,16.56%
KEY,-6.09%,-0.690,1.322,8.506,0.380,27.28%
PNC,-2.38%,-0.380,1.173,10.601,0.488,19.42%
TFC,-6.70%,-0.950,1.134,9.125,0.414,21.82%
USB,-7.02%,-1.062,1.060,9.098,0.412,20.45%


### Step 4 Interpretation

I would not treat any alpha as statistically reliable because none has an absolute alpha t-statistic of at least about 1.98, the usual 5% significance threshold. This does not mean the banks performed poorly: several had positive returns and positive Sharpe ratios, but their returns were not reliably higher than CAPM would predict given their market risk. The EW panel’s alpha t-statistic is also not significant because, although diversification lowered residual volatility, its estimated alpha was still small relative to the uncertainty around that estimate.

In [10]:
market_ann_vol = mkt_excess.std(ddof=1) * np.sqrt(MONTHS_PER_YEAR)

step5 = pd.DataFrame(index=excess_all.columns)
step5["β"] = step4["β"]

# This must match the covariance-matrix diagonal later
step5["total"] = excess_all.std(ddof=1) * np.sqrt(MONTHS_PER_YEAR)

step5["systematic"] = step5["β"].abs() * market_ann_vol
step5["idiosyncratic"] = step4["resid vol"]

# Variance share explained by systematic market risk
step5["share"] = (step5["systematic"] ** 2) / (step5["total"] ** 2)

step5_display = step5.copy()
step5_display["β"] = step5_display["β"].map(lambda x: f"{x:.3f}")
for col in ["total", "systematic", "idiosyncratic", "share"]:
    step5_display[col] = step5_display[col].map(lambda x: f"{x:.1%}")

step5_display

,β,total,systematic,idiosyncratic,share
BAC,1.415,29.7%,22.8%,19.2%,58.6%
C,1.532,32.6%,24.7%,21.4%,57.3%
CFG,1.461,36.5%,23.5%,28.0%,41.6%
IBM,0.889,24.0%,14.3%,19.3%,35.6%
JPM,1.126,24.5%,18.1%,16.6%,54.7%
KEY,1.322,34.5%,21.3%,27.3%,38.0%
PNC,1.173,27.0%,18.9%,19.4%,48.8%
TFC,1.134,28.4%,18.3%,21.8%,41.4%
USB,1.060,26.6%,17.1%,20.4%,41.2%
WFC,1.156,29.3%,18.6%,22.8%,40.2%


In [11]:
check_name = "AAPL" if "AAPL" in step5.index else security_cols[0]

reconstructed_total = np.sqrt(
    step5.loc[check_name, "systematic"] ** 2
    + step5.loc[check_name, "idiosyncratic"] ** 2
)

print(
    f"check {check_name}: sqrt("
    f"{step5.loc[check_name, 'systematic']:.2%}² + "
    f"{step5.loc[check_name, 'idiosyncratic']:.2%}²) = "
    f"{reconstructed_total:.2%}"
)
print(f"reported total volatility = {step5.loc[check_name, 'total']:.2%}")

check BAC: sqrt(22.77%² + 19.22%²) = 29.80%
reported total volatility = 29.75%


In [12]:
median_single_idio = step5.loc[security_cols, "idiosyncratic"].median()
ew_idio = step5.loc["EW panel", "idiosyncratic"]

print(f"EW panel idiosyncratic volatility: {ew_idio:.2%}")
print(f"Median single-name idiosyncratic volatility: {median_single_idio:.2%}")

EW panel idiosyncratic volatility: 18.07%
Median single-name idiosyncratic volatility: 21.40%


# Step 5

The equal-weight panel’s idiosyncratic volatility is 18.07% compared with a median single-name idiosyncratic volatility of 21.40%. This shows that equal-weighting across 11 securities materially reduced company-specific risk, although it did not remove market-wide systematic risk as all the securities were banks that share similar market exposure

In [13]:
# Keep original security order from Part 1
week4_excess_returns = asset_excess[security_cols].copy()
week4_cov = week4_excess_returns.cov() * MONTHS_PER_YEAR

returns_file = Path("week4_monthly_excess_returns.csv")
cov_file = Path("week4_annualized_excess_covariance.csv")

week4_excess_returns.to_csv(returns_file, index_label="Date")
week4_cov.to_csv(cov_file, index_label="Security")

# Reload and confirm the order is preserved
returns_check = pd.read_csv(returns_file, index_col="Date")
cov_check = pd.read_csv(cov_file, index_col="Security")

assert returns_check.columns.tolist() == security_cols
assert cov_check.index.tolist() == security_cols
assert cov_check.columns.tolist() == security_cols

print("Week 4 files saved and column order verified.")
print("\n3×3 covariance-matrix corner:")
display(week4_cov.iloc[:3, :3])

Week 4 files saved and column order verified.

3×3 covariance-matrix corner:


,BAC,C,CFG
BAC,0.088483,0.084526,0.095079
C,0.084526,0.106193,0.095944
CFG,0.095079,0.095944,0.132888


In [14]:
diag_check = pd.DataFrame({
    "covariance diagonal": np.diag(week4_cov),
    "Step 5 total vol squared": step5.loc[security_cols, "total"] ** 2
}, index=security_cols)

diag_check["difference"] = (
    diag_check["covariance diagonal"] - diag_check["Step 5 total vol squared"]
)

display(diag_check.head())

assert np.allclose(
    diag_check["covariance diagonal"],
    diag_check["Step 5 total vol squared"]
)

print("Diagonal check passed: covariance diagonal equals Step 5 total volatility squared.")

,covariance diagonal,Step 5 total vol squared,difference
BAC,0.088483,0.088483,-1.387779e-17
C,0.106193,0.106193,5.551115e-17
CFG,0.132888,0.132888,0.000000e+00
IBM,0.057432,0.057432,6.938894e-18
JPM,0.060048,0.060048,1.387779e-17


Diagonal check passed: covariance diagonal equals Step 5 total volatility squared.


**Stopping point:** Completed Steps 3–5, including the saved and verified Week 4 excess-return and covariance-matrix inputs.

**AI disclosure:** AI was used to help explain the required calculations and write Python code. I reviewed, ran, and verified all code and the interpretations myself. I made sure to review the output and ensure that the values made sense based on the given securities and were consistent throughout each step. 